<a id="0"></a>
# <div style="padding:20px;color:white;margin:0;font-size:30px;font-family:Georgia;text-align:center;display:fill;border-radius:5px;background-color:#255E58;overflow:hidden">Análise Avançada de Dados para IoT  📊📈</div>

<a id="1"></a>
# <div style="padding:20px;color:white;margin:0;font-size:30px;font-family:Georgia;text-align:center;display:fill;border-radius:5px;background-color:#254E58;overflow:hidden">Módulo 8 → Clusterização e Detecção de Anomalias
</div>

<a id="1"></a>
<h2 style="font-family: Verdana; font-size: 25px; font-style: normal; font-weight: normal; text-decoration: none; text-transform: none; letter-spacing: 2px; color: #254E58; background-color: #ffffff;"><b> Vídeo</b> 1

> <span style='font-size:15px; font-family:Verdana;color: #c6dadfff; border-bottom: solid 3px'><b>Importando
Bibliotecas</b></span>

In [ ]:
# BIBLIOTECAS
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, accuracy_score, confusion_matrix
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.ensemble import IsolationForest
from sklearn.tree import DecisionTreeClassifier
from sklearn.pipeline import Pipeline
from sklearn.decomposition import PCA


# CONFIGURAÇÕES DE ESTÉTICA👇🏽

# Ajustes visuais do Jupyter
from IPython.core.display import HTML
display(HTML('<style>.container { width:80% !important; }</style>'))
# pd.set_option('display.max_columns', 60)

# Configurando pandas para mostrar todas as linhas e colunas
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None) 

# Configurando pandas para não mostrar notação científica para números
pd.set_option('display.float_format', lambda x: '%.2f' % x)
# Serve para Manter os gráficos nesse mesmo Notebook
%matplotlib inline
# Ignorando alertas
import warnings
warnings.filterwarnings("ignore")

# Reprodutibilidade
np.random.seed(42)

> <span style='font-size:15px; font-family:Verdana;color: #c6dadfff; border-bottom: solid 3px'><b>Gerando os Dados e Criando o DataFrame</b></span>

In [ ]:
# GERANDO DADOS
np.random.seed(42)
n = 300

temperatura = np.random.normal(30, 5, n)

dados = pd.DataFrame({
    "temperatura": temperatura,
    "vibracao": 0.12 * temperatura + np.random.normal(0, 0.8, n),
    "pressao": 100 + 0.7 * temperatura + np.random.normal(0, 3, n),
    "corrente": 8 + 0.15 * temperatura + np.random.normal(0, 1, n),
    "umidade": 70 - 0.6 * temperatura + np.random.normal(0, 4, n),
    "rpm": 1200 + 18 * temperatura + np.random.normal(0, 80, n)
})

# PADRONIZAÇÃO
X = StandardScaler().fit_transform(dados)

# PCA
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X)

print(f"PC1: {pca.explained_variance_ratio_[0] * 100:.2f}%")
print(f"PC2: {pca.explained_variance_ratio_[1] * 100:.2f}%")
print(f"Total: {pca.explained_variance_ratio_.sum() * 100:.2f}%")

# GRÁFICO PCA
plt.figure(figsize=(8, 5))
plt.scatter(X_pca[:, 0], X_pca[:, 1])
plt.title("PCA: 6 sensores reduzidos para 2 dimensões")
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.grid()
plt.show()

# VARIÂNCIA ACUMULADA
pca_total = PCA().fit(X)
variancia = np.cumsum(pca_total.explained_variance_ratio_) * 100

plt.figure(figsize=(8, 5))
plt.plot(range(1, len(variancia) + 1), variancia, marker="o")
plt.title("Informação preservada pelo PCA")
plt.xlabel("Número de componentes")
plt.ylabel("Variância explicada acumulada (%)")
plt.grid()
plt.show()

In [ ]:
dados.head()

<a id="1"></a>
<h2 style="font-family: Verdana; font-size: 25px; font-style: normal; font-weight: normal; text-decoration: none; text-transform: none; letter-spacing: 2px; color: #254E58; background-color: #ffffff;"><b> Vídeo</b> 2

In [ ]:
def gerar_dados(n=60):
    np.random.seed(42)

    return pd.concat([
        pd.DataFrame({
            "temperatura": np.random.normal(28, 1.0, n),
            "vibracao": np.random.normal(2.0, 0.25, n)
        }),
        pd.DataFrame({
            "temperatura": np.random.normal(33, 1.0, n),
            "vibracao": np.random.normal(3.5, 0.30, n)
        }),
        pd.DataFrame({
            "temperatura": np.random.normal(38, 1.1, n),
            "vibracao": np.random.normal(5.0, 0.35, n)
        })
    ], ignore_index=True)


def aplicar_kmeans(df):
    X = df[["temperatura", "vibracao"]]

    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    modelo = KMeans(n_clusters=3, random_state=42, n_init=10)
    df["cluster"] = modelo.fit_predict(X_scaled)

    centros = scaler.inverse_transform(modelo.cluster_centers_)

    ordem = np.argsort(centros[:, 0])
    nomes = {
        ordem[0]: "Operação normal",
        ordem[1]: "Carga média",
        ordem[2]: "Alta carga"
    }

    df["regime"] = df["cluster"].map(nomes)

    return centros


def plotar_clusters(df, centros):
    plt.figure(figsize=(8, 5))

    for regime in ["Operação normal", "Carga média", "Alta carga"]:
        parte = df[df["regime"] == regime]

        plt.scatter(
            parte["temperatura"],
            parte["vibracao"],
            alpha=0.75,
            label=regime
        )

    plt.scatter(
        centros[:, 0],
        centros[:, 1],
        marker="X",
        s=220,
        edgecolor="black",
        label="Centros"
    )

    plt.title("Clusterização de Sensores IoT com K-Means")
    plt.xlabel("Temperatura (°C)")
    plt.ylabel("Vibração")
    plt.legend()
    plt.grid(alpha=0.3)
    plt.show()


df = gerar_dados()
centros = aplicar_kmeans(df)
plotar_clusters(df, centros)

<a id="1"></a>
<h2 style="font-family: Verdana; font-size: 25px; font-style: normal; font-weight: normal; text-decoration: none; text-transform: none; letter-spacing: 2px; color: #254E58; background-color: #ffffff;"><b> Vídeo</b> 3

In [ ]:
def transformar(dados):
    normalizado = MinMaxScaler().fit_transform(dados)
    padronizado = StandardScaler().fit_transform(dados)

    print("Originais:", dados.ravel())
    print("Normalizados:", normalizado.ravel().round(2))
    print("Padronizados:", padronizado.ravel().round(2))
    print("Média:", round(padronizado.mean(), 2))
    print("Desvio padrão:", round(padronizado.std(), 2))


dados = np.array([[10], [20], [30], [40], [50]])

transformar(dados)

<a id="1"></a>
<h2 style="font-family: Verdana; font-size: 25px; font-style: normal; font-weight: normal; text-decoration: none; text-transform: none; letter-spacing: 2px; color: #254E58; background-color: #ffffff;"><b> Vídeo</b> 4

In [ ]:
def gerar_dados(n=60):
    np.random.seed(42)

    normal = pd.DataFrame({
        "temperatura": np.random.normal(28, 1.0, n),
        "vibracao": np.random.normal(2.0, 0.25, n)
    })

    carga_media = pd.DataFrame({
        "temperatura": np.random.normal(33, 1.0, n),
        "vibracao": np.random.normal(3.5, 0.30, n)
    })

    alta_carga = pd.DataFrame({
        "temperatura": np.random.normal(38, 1.1, n),
        "vibracao": np.random.normal(5.0, 0.35, n)
    })

    return pd.concat(
        [normal, carga_media, alta_carga],
        ignore_index=True
    )


def metodo_cotovelo(X_scaled):
    inercias = []

    for k in range(1, 7):
        modelo = KMeans(
            n_clusters=k,
            random_state=42,
            n_init=10
        )

        modelo.fit(X_scaled)
        inercias.append(modelo.inertia_)

    plt.figure(figsize=(7, 4))
    plt.plot(range(1, 7), inercias, marker="o")

    plt.title("Método do Cotovelo")
    plt.xlabel("Número de clusters (k)")
    plt.ylabel("Inércia")
    plt.grid(alpha=0.3)

    plt.show()


def aplicar_kmeans(df):
    X = df[["temperatura", "vibracao"]]

    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    kmeans = KMeans(
        n_clusters=3,
        random_state=42,
        n_init=10
    )

    df["cluster"] = kmeans.fit_predict(X_scaled)

    centros = scaler.inverse_transform(
        kmeans.cluster_centers_
    )

    ordem = np.argsort(centros[:, 0])

    nomes = {
        ordem[0]: "Operação normal",
        ordem[1]: "Carga média",
        ordem[2]: "Alta carga"
    }

    df["regime"] = df["cluster"].map(nomes)

    return df, X_scaled, centros


def plotar_clusters(df, centros):
    plt.figure(figsize=(8, 5))

    for regime in [
        "Operação normal",
        "Carga média",
        "Alta carga"
    ]:
        parte = df[df["regime"] == regime]

        plt.scatter(
            parte["temperatura"],
            parte["vibracao"],
            alpha=0.75,
            label=regime
        )

    plt.scatter(
        centros[:, 0],
        centros[:, 1],
        marker="X",
        s=220,
        edgecolor="black",
        label="Centros"
    )

    plt.title("Clusterização de Sensores IoT com K-Means")
    plt.xlabel("Temperatura (°C)")
    plt.ylabel("Vibração")
    plt.legend()
    plt.grid(alpha=0.3)

    plt.show()


# GERANDO OS DADOS
df = gerar_dados()

# PREPARANDO OS DADOS PARA ENTREGAR AO K-MEANS
df, X_scaled, centros = aplicar_kmeans(df)

# MÉTODO DO COTOVELO
metodo_cotovelo(X_scaled)

# GRÁFICO DOS CLUSTERS
plotar_clusters(df, centros)

# RESULTADOS
resumo = (
    df.groupby("regime")[["temperatura", "vibracao"]]
    .mean()
    .round(2)
)

print(resumo)

<a id="1"></a>
<h2 style="font-family: Verdana; font-size: 25px; font-style: normal; font-weight: normal; text-decoration: none; text-transform: none; letter-spacing: 2px; color: #254E58; background-color: #ffffff;"><b> Vídeo</b> 5

In [ ]:
# GERANDO OS DADOS
np.random.seed(42)
n = 180

df = pd.DataFrame({
    "tempo": pd.date_range("2026-09-27 08:00", periods=n, freq="1min"),
    "temperatura": np.random.normal(30, 1.0, n),
    "vibracao": np.random.normal(2.5, 0.25, n)
})

idx = [35, 90, 140, 165]

df.loc[idx, "temperatura"] += [8, -7, 10, -6]
df.loc[idx, "vibracao"] += [2.0, 1.8, 2.5, 2.2]

# ISOLATION FOREST
X = df[["temperatura", "vibracao"]]

modelo = IsolationForest(
    contamination=0.03,
    random_state=42
)

df["predicao"] = modelo.fit_predict(X)
df["status"] = np.where(df["predicao"] == -1, "Anomalia", "Normal")

anom = df[df["status"] == "Anomalia"]
normais = df[df["status"] == "Normal"]

# GRÁFICO TEMPORAL
plt.figure(figsize=(11, 4))
plt.plot(df["tempo"], df["temperatura"], label="Temperatura")
plt.scatter(anom["tempo"], anom["temperatura"], s=90, label="Anomalias")

plt.title("Isolation Forest - Anomalias na Temperatura")
plt.xlabel("Tempo")
plt.ylabel("Temperatura (°C)")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

# RELAÇÃO ENTRE----> NORMAL x ANOMALIA
plt.figure(figsize=(8, 5))
plt.scatter(normais["temperatura"], normais["vibracao"], alpha=0.6, label="Normal")
plt.scatter(anom["temperatura"], anom["vibracao"], s=100, label="Anomalia")

plt.title("Isolation Forest - Normal x Anomalia")
plt.xlabel("Temperatura (°C)")
plt.ylabel("Vibração")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

# VISUALIZANDO OS RESULTADOS
print("Registros:", len(df))
print("Anomalias:", len(anom))
print(anom[["tempo", "temperatura", "vibracao"]].round(2))